# 01 — Data Exploration

This notebook walks through the LOB-Forge data layer:
- LOB schema and column layout
- Feature engineering (OFI, MLOFI, VPIN, microprice)
- Temporal splits with purge gap
- Data quality validation

All cells run without real data files — dummy arrays are used where actual parquet files are absent.

In [ ]:
import matplotlib
import numpy as np
import pandas as pd

matplotlib.use('Agg')

import matplotlib.pyplot as plt

np.random.seed(42)
print('Dependencies loaded.')

## 1. LOB Schema

The 46-column LOB format used throughout LOB-Forge:
- **Header (3):** timestamp, trade_price, trade_size
- **Ask book (20):** ask_price_1..10, ask_size_1..10
- **Bid book (20):** bid_price_1..10, bid_size_1..10
- **Label (3):** mid, label_1s, label_2s, ... (added during preprocessing)

In the 40-column evaluation layout (book-only):
- cols 0-9: ask_price_1..10
- cols 10-19: ask_size_1..10
- cols 20-29: bid_price_1..10
- cols 30-39: bid_size_1..10

In [ ]:
from lob_forge.data.schema import ALL_COLUMNS, LOB_SCHEMA

# LOB_SCHEMA is a pyarrow.Schema object
n_fields = len(LOB_SCHEMA)
n_cols = len(ALL_COLUMNS)
print('Schema fields:', n_fields)
print('All columns count:', n_cols)
print('First 10 columns:', ALL_COLUMNS[:10])
print('LOB_SCHEMA fields (sample):')
for i in range(min(8, len(LOB_SCHEMA))):
    field = LOB_SCHEMA.field(i)
    print(' ', field.name + ':', str(field.type))


In [ ]:
# Show column layout as a DataFrame summary
groups = {
    'Header': ['timestamp', 'mid_price', 'spread'],
    'Bid Price (1-10)': [f'bid_price_{i}' for i in range(1, 11)],
    'Bid Size (1-10)':  [f'bid_size_{i}' for i in range(1, 11)],
    'Ask Price (1-10)': [f'ask_price_{i}' for i in range(1, 11)],
    'Ask Size (1-10)':  [f'ask_size_{i}' for i in range(1, 11)],
    'Trade': ['trade_price', 'trade_size', 'trade_side'],
}

rows = []
for group, cols in groups.items():
    rows.append({'Group': group, 'Count': len(cols), 'Example columns': ', '.join(cols[:3]) + ('...' if len(cols) > 3 else '')})

pd.DataFrame(rows)


## 2. Feature Engineering

LOB-Forge computes 20 engineered features from raw book data:

| Feature | Description |
|---------|-------------|
| **OFI** | Order Flow Imbalance — signed volume at best bid/ask |
| **MLOFI** | Multi-Level OFI aggregated across 10 price levels |
| **VPIN** | Volume-synchronised PIN — adverse selection proxy |
| **Microprice** | Inventory-weighted mid-price |
| **Spread bps** | Bid-ask spread in basis points |
| **Depth imbalance** | Total ask vs bid depth ratio |
| **Mid returns** | Log-returns of mid-price series |

In [ ]:
import inspect

from lob_forge.data.features import (
    compute_all_features,
    compute_vpin,
)

print('compute_all_features signature:')
print(' ', inspect.signature(compute_all_features))
print()
print('compute_vpin signature:')
print(' ', inspect.signature(compute_vpin))

In [ ]:
# Build a dummy LOB DataFrame (46 columns) and run feature engineering
N = 500
base_price = 100.0

data = {}
data['timestamp'] = pd.date_range('2024-01-01', periods=N, freq='1s').astype('int64') // 10**9
data['mid_price'] = base_price + np.random.randn(N).cumsum() * 0.01
data['spread'] = np.abs(np.random.randn(N)) * 0.01 + 0.005
data['trade_price'] = base_price + np.random.randn(N).cumsum() * 0.01
data['trade_size'] = np.random.randint(1, 100, N).astype(float)
data['trade_side'] = np.zeros(N)

for i in range(1, 11):
    data[f'bid_price_{i}'] = base_price - i * 0.01 + np.random.randn(N) * 0.001
    data[f'bid_size_{i}'] = np.random.randint(10, 500, N).astype(float)
    data[f'ask_price_{i}'] = base_price + i * 0.01 + np.random.randn(N) * 0.001
    data[f'ask_size_{i}'] = np.random.randint(10, 500, N).astype(float)

df_raw = pd.DataFrame(data)
print('Raw LOB shape:', df_raw.shape)

from lob_forge.data.features import compute_all_features

df_features = compute_all_features(df_raw)
print('After feature engineering:', df_features.shape)
feature_cols = [c for c in df_features.columns if c not in df_raw.columns]
print('New feature columns (', len(feature_cols), '):', feature_cols)


In [ ]:
# Plot feature distributions
feature_cols = [c for c in df_features.columns if c not in df_raw.columns]
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, col in enumerate(feature_cols[:8]):
    ax = axes[i // 4, i % 4]
    vals = df_features[col].dropna()
    ax.hist(vals, bins=40, color='#4C72B0', edgecolor='white', alpha=0.8)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
plt.suptitle('Feature Distributions (dummy data)', fontsize=12)
plt.tight_layout()
plt.savefig('/tmp/feature_distributions.png', dpi=100)
plt.close()
print('Feature distribution plot saved.')

## 3. Temporal Splits

LOB-Forge uses a **purge-gap** temporal split to avoid look-ahead bias:

```
  |<---- train ---->|<-- purge -->|<-- val -->|<-- purge -->|<-- test -->|
                    |    gap      |           |    gap      |
```

The purge gap (default 10 rows in production) discards rows near split boundaries where labels could leak information across the boundary — this is critical for LOB data where label at row `t` uses future mid-prices `t+1..t+h`.

In [ ]:
import inspect

from lob_forge.data.splits import temporal_split

print('temporal_split signature:')
print(' ', inspect.signature(temporal_split))
print()

# Demonstrate temporal split on dummy index
n_rows = 1000
train_idx, val_idx, test_idx = temporal_split(
    n_rows,
    ratios=(0.7, 0.15, 0.15),
    purge_gap=10,
)

print('Total rows: ', n_rows)
print('Train rows:', len(train_idx), '(', round(100*len(train_idx)/n_rows, 1), '%)')
print('Val rows:  ', len(val_idx),   '(', round(100*len(val_idx)/n_rows, 1), '%)')
print('Test rows: ', len(test_idx),  '(', round(100*len(test_idx)/n_rows, 1), '%)')
print('Purged:    ', n_rows - len(train_idx) - len(val_idx) - len(test_idx))


In [ ]:
# ASCII diagram of the split
total = 80
n = 1000
t_end = int(train_idx[-1]) if len(train_idx) else 0
v_start = int(val_idx[0]) if len(val_idx) else t_end + 10
v_end = int(val_idx[-1]) if len(val_idx) else v_start
te_start = int(test_idx[0]) if len(test_idx) else v_end + 10

def to_char(row, total=80, n=1000):
    return int(row / n * total)

train_end_c = to_char(t_end)
val_start_c = to_char(v_start)
val_end_c = to_char(v_end)
test_start_c = to_char(te_start)

line = ''
for i in range(total):
    if i < train_end_c:
        line += 'T'
    elif i < val_start_c:
        line += '-'
    elif i < val_end_c:
        line += 'V'
    elif i < test_start_c:
        line += '-'
    else:
        line += 'E'

print('Temporal split layout (T=train, V=val, E=test, -=purge gap):')
print(f'|{line}|')
print(f' {"Train":^{train_end_c}}{"Purge":^{val_start_c-train_end_c}}{"Val":^{val_end_c-val_start_c}}{"Purge":^{test_start_c-val_end_c}}{"Test":^{total-test_start_c}}')

## 4. Data Quality Checks

`validate_lob_dataframe` checks:
- Required columns present
- No NaN in book columns (ask/bid price+size)
- Monotonic ask prices (ask_1 < ask_2 < ... < ask_10)
- Monotonic bid prices (bid_1 > bid_2 > ... > bid_10)
- Positive spread (ask_1 > bid_1)
- Positive sizes

In [ ]:
import inspect

from lob_forge.data.validation import compute_quality_metrics, validate_lob_dataframe

print('validate_lob_dataframe signature:')
print(' ', inspect.signature(validate_lob_dataframe))
print()
print('compute_quality_metrics signature:')
print(' ', inspect.signature(compute_quality_metrics))

In [ ]:
# Run validation on the dummy LOB DataFrame
issues = validate_lob_dataframe(df_raw)
if issues:
    print(f'Validation issues found ({len(issues)}):')
    for issue in issues:
        print(f'  - {issue}')
else:
    print('No validation issues — dummy data passes all checks.')

# Quality metrics
metrics = compute_quality_metrics(df_raw)
print('\nQuality metrics:')
for k, v in metrics.items():
    print(f'  {k}: {v}')

In [ ]:
# Introduce a deliberate error and validate
df_bad = df_raw.copy()
df_bad.loc[0, 'ask_price_1'] = np.nan

issues_bad = validate_lob_dataframe(df_bad)
print(f'Issues in corrupt DataFrame: {len(issues_bad)}')
for issue in issues_bad:
    print(f'  - {issue}')